In [ ]:
library(Pigengene)
library(ggplot2)

In [ ]:
library(Seurat)

In [ ]:
path2res <- "/home/xinyuelu/AD-proj/results/results-annotate-hPFC/senescence"

## Load senescence marker

In [ ]:
gene.set.1 <- c("CDKN2D", "ETS2", "RB1", "E2F3", "CDK6", "RBL2", "ATM", "BMI1", 
           "MDM2", "CDK4", "CCNE1", "E2F1", "CHEK2", "CHEK1", "CDKN1A", 
           "TWIST1", "CCND1", "ETS1", "TP53", "CDKN2A", "CDK2", "SATB1")
gene.set.2 <- c("SOD1", "MAP2K1", "GSK3B", "PIK3CA", "SOD2", "MAPK14", "IGF1R", "TP53BP1", 
           "NBN", "HRAS", "CITED2", "CREG1", "ABL1", "MORC3", "NFKB1", "AKT1", 
           "CDKN1B", "EGR1", "RBL1", "MAP2K6", "IGF1", "IRF3", "PCNA", "GADD45A", 
           "MAP2K3", "IGFBP5", "SIRT1", "ING1", "TGFB1", "TERF2", "CCNB1", "PRKCD", 
           "CDC25C", "IGFBP3", "ALDH1A3", "MYC", "NOX4", "CCNA2", "CDKN2C", "TERT", 
           "ID1", "IGFBP7", "CDKN1C", "IRF7", "IFNG", "CDKN2B", "PLAU", "IRF5")
gene.set.3 <- c("IGFBP7", "VIM", "FN1", "SPARC", "IGFBP4", "TIMP1", "TBX2", "TBX3", 
           "COL1A1", "COL3A1", "IGFBP2", "TGFB1I1", "PTEN", "CD44", "NFIA", "CALR", 
           "TIMP2", "CXCL8", "IL6", "FGF2", "FGF7", "AKT1", "CXCL2", "VEGFA", 
           "CXCL1", "PLAUR", "SERPINE1", "LMNB1", "GLB1", "VEGFB", "CCL2", "IL1B", 
           "CXCL5", "SERPINB2", "IL11", "IL1A", "CCL5", "TNF", "CCL20", "MMP1", 
           "MMP3", "MMP12", "MMP10")

In [ ]:
all.genes <- unique(c(gene.set.1, gene.set.2, gene.set.3))
gene.vector <- setNames(rep(NA, length(all.genes)), all.genes)

gene.vector[gene.set.1] <- 0
gene.vector[gene.set.2] <- 1
gene.vector[gene.set.3] <- 2

## Test on Allen Brain data

In [ ]:
library(Seurat)
library(tibble)

expr <- read.csv("/home/xinyuelu/AD-proj/reference/DLPFC_expression_data.csv", row.names = 1)
expr <- as.matrix(t(expr))

cell_meta <- read.csv("/home/xinyuelu/AD-proj/reference/DLPFC_cell_metadata.csv", row.names = 1)
cell_meta <- as.data.frame(cell_meta)

gene_meta <- read.csv("/home/xinyuelu/AD-proj/reference/DLPFC_gene_metadata.csv", row.names = 1)
gene_meta <- as.data.frame(gene_meta)


## Load GAGE-seq RNA and Spatial data

In [ ]:
obj.gage <- readRDS("/home/xinyuelu/AD-proj/results/results-annotate-hPFC/annotation/AD_rna_annotated.rds")

In [ ]:
obj.spatial <- readRDS("/home/xinyuelu/AD-proj/spatial/spatial_only_AD.rds")

In [ ]:
gc()

In [ ]:
obj.gage

In [ ]:
obj.spatial

In [ ]:
obj.gage.micro <- subset(obj.gage, idents = 'Micro')
obj.spatial.micro <- subset(obj.spatial, idents = 'Micro')

In [ ]:
obj.gage.micro

In [ ]:
obj.spatial.micro

In [ ]:
saved <- options(repr.plot.width=12, repr.plot.height=6)
DimPlot(obj.gage.micro, group.by = 'condition') +
DimPlot(obj.gage.micro, group.by = 'sample_id')
options(saved)

In [ ]:
saved <- options(repr.plot.width=12, repr.plot.height=6)
DimPlot(obj.spatial.micro, group.by = 'condition') +
DimPlot(obj.spatial.micro, group.by = 'sample_id')
options(saved)

In [ ]:
umap_coords <- Embeddings(obj.spatial.micro, reduction = "umap")
selected_cells <- rownames(umap_coords)[umap_coords[,'umap_1'] > 2 & umap_coords[,'umap_2'] > 0]
obj.spatial.micro <- subset(obj.spatial.micro, cells = selected_cells)

In [ ]:
obj.spatial.micro <- RunUMAP(obj.spatial.micro, dims = 1:20, n.neighbors = 20, min.dist = 0.1)
saved <- options(repr.plot.width=12, repr.plot.height=6)
DimPlot(obj.spatial.micro, group.by = 'condition') +
DimPlot(obj.spatial.micro, group.by = 'sample_id')
options(saved)

In [ ]:
final_coords <- as.data.frame(Embeddings(obj.spatial.micro, reduction = "umap"))

In [ ]:
cells_micro1 <- rownames(final_coords)[final_coords$umap_1 < 2]
obj.spatial@meta.data[cells_micro1, "major_cell_type"] <- "Micro1"

In [ ]:
FeaturePlot(obj.spatial.micro, features = "CDKN1A", reduction = "umap", split.by = "condition",
            min.cutoff = "q10", max.cutoff = "q90")

### Load color map

In [ ]:
major_base_colors <- c(
    "Exc" = "turquoise4", 
    "Inh" = "deeppink2", 
    "Astro" = "orchid4",
    "Oligo" = "yellow3", 
    "Micro" = "steelblue4", 
    "OPC" = "springgreen4",
    "Endo" = "bisque4", 
    "VLMC" = "lightsalmon"
)

In [ ]:
sub_base_colors <- c(
    "Exc L2/3 IT" = "turquoise4", 
    "Exc L4 IT" = "lightskyblue3",
    "Exc L5 IT" = "lightgreen",
    "Exc L5 ET" = "cyan4",
    "Exc L5/6 NP" = "seagreen2",
    "Exc L6 IT" = "skyblue4",
    "Exc L6b" = "royalblue1",
    "Exc L6 CT" = "lightskyblue1",
    "Exc L6 IT Car3" = "cornflowerblue",
    "Inh Lamp5" = "deeppink4", 
    "Inh Sst" = "lightpink3",
    "Inh Sncg" = "firebrick4",
    "Inh Pvalb" = "tomato1",
    "Inh Vip" = "deeppink2",
    "Inh PAX6" = "lightpink1",
    "Inh Chandelier" = "plum1",
    "Oligo" = "yellow3", 
    "Astro" = "orchid4",
    "Micro" = "steelblue4", 
    "OPC" = "springgreen4",
    "Endo" = "bisque4", 
    "VLMC" = "palegreen3"
)

### Fit model on RNA dataset

In [ ]:
expr_matrix <- as.matrix(GetAssayData(obj.gage, slot = "data"))
cell_types <- obj.gage@meta.data$sub_cell_type
expr_matrix_f <- expr_matrix[rownames(expr_matrix) %in% names(gene.vector), , drop = FALSE]
expr_matrix_f <- t(expr_matrix_f)
dim(expr_matrix_f)

In [ ]:
Labels <- setNames(obj.gage@meta.data$sub_cell_type, rownames(obj.gage@meta.data))
stages <- obj.gage$condition

In [ ]:
f.gene.vector <- gene.vector[names(gene.vector) %in% rownames(obj.gage)]

In [ ]:
savefile <- file.path(path2res, "pigengene0-rna-all.RData")
eigengenes0 <- compute.pigengene(Data=expr_matrix_f, 
                                 modules=f.gene.vector, 
                                 Labels = Labels, 
                                 selectedModules = 0, 
                                 doPlot=FALSE,
                                 saveFile=savefile)
gc()

In [ ]:
savefile <- file.path(path2res, "pigengene1-rna-all.RData")
eigengenes1 <- compute.pigengene(Data=expr_matrix_f, 
                                 modules=f.gene.vector, 
                                 Labels = Labels, 
                                 selectedModules = 1, 
                                 doPlot=FALSE,
                                 saveFile=savefile)
gc()

In [ ]:
savefile <- file.path(path2res, "pigengene2-rna-all.RData")
eigengenes2 <- compute.pigengene(Data=expr_matrix_f, 
                                 modules=f.gene.vector, 
                                 Labels = Labels, 
                                 selectedModules = 2, 
                                 doPlot=FALSE,
                                 saveFile=savefile)
gc()

### Load RNA results

In [ ]:
savefile <- file.path(path2res, "pigengene0-rna-all.RData")
load(savefile)
pigengene0_rna <- pigengene
savefile <- file.path(path2res, "pigengene1-rna-all.RData")
load(savefile)
pigengene1_rna <- pigengene
savefile <- file.path(path2res, "pigengene2-rna-all.RData")
load(savefile)
pigengene2_rna <- pigengene
gc()

In [ ]:
savefile <- "/home/xinyuelu/AD-proj/scripts_results/annotate_cells/pigengene0-AD.RData"
load(savefile)
pigengene0_ad <- pigengene
savefile <- "/home/xinyuelu/AD-proj/scripts_results/annotate_cells/pigengene0-CT.RData"
load(savefile)
pigengene0_ct <- pigengene
gc()

### Load spatial results

In [ ]:
load("/home/xinyuelu/AD-proj/scripts_results/annotate_cells/pigengene0-spatial-all.RData")
pigengene0 <- pigengene
load("/home/xinyuelu/AD-proj/scripts_results/annotate_cells/pigengene1-spatial-all.RData")
pigengene1 <- pigengene
load("/home/xinyuelu/AD-proj/scripts_results/annotate_cells/pigengene2-spatial-all.RData")
pigengene2 <- pigengene
gc()

## Process pigengene results

In [ ]:
plot_gene_weight <- function(df, x_col = "Name", y_col = "Weight", title = "Module Weights", savename = "") {
  
  if (!(x_col %in% names(df)) | !(y_col %in% names(df))) {
    stop("Columns not found in the dataframe.")
  }
  
  p <- ggplot(df, aes_string(x = x_col, y = y_col, fill = y_col)) +
    geom_bar(stat = "identity", width = 0.8) +
    scale_fill_gradient(low = "darkorchid3", high = "darkorange2") +  
    theme_minimal(base_size = 16) +  
    labs(x = "", y = "Weights", title = title) +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 16, face = "bold", color = "black"),  
      axis.text.y = element_text(size = 18, face = "bold", color = "black"),
      axis.title.y = element_text(size = 22, face = "bold", color = "black"),
      plot.title = element_text(size = 26, hjust = 0.5, face = "bold", color = "black"),
      legend.position = "none" 
    )
  
  ggsave(paste0("weights_eigen_", savename, ".pdf"), plot = p, width = 25, height = 15, dpi = 350)
  print(p)
}

In [ ]:
process_eigengene = function(pigengene, module, res_path, gene.set, label, stages, thres=3) {
    e1 <-  pigengene$eigengenes[,paste0("ME", module)]
    names(e1) <- rownames(pigengene$eigengenes)
    print(e1[0:5])

    meModule <- sort(pigengene$membership[gene.set, ],
                     decreasing=TRUE)
    print(paste("SD(e1):", sd(e1)))
    print(summary(e1))

    df <- data.frame(Name = names(meModule), Weight = meModule)

    pop <- cbind(label[names(e1)], as.data.frame(e1)) 
    colnames(pop) <- c("Cell_type", "Expression")
    pop[, "Cell_type"] <- factor(pop[, "Cell_type"])
    pop <- cbind(pop, Population= e1[rownames(pop)]> mean(e1)+thres*sd(e1))
    pop <- cbind(pop, Stage= stages[rownames(pop)])
    pop$Label <- paste(pop$Cell_type, pop$Stage, sep = "-")
    numbersCt <- table(pop[,c("Label", "Population")])

    ratioCt <- numbersCt[,"TRUE"]/rowSums(numbersCt)
    numbersCt <- cbind(numbersCt, ratioCt)
    numbersCt <- cbind(numbersCt, "P"= 0)
    numbersCt <- cbind(numbersCt, "P(not)"= 0)

    for(type1 in rownames(numbersCt)){
        p1 <- phyper(m= sum(numbersCt[,"TRUE"]), n= sum(numbersCt[,"FALSE"]),
                     k=sum(pop[,"Label"]==type1), q=numbersCt[type1,"TRUE"],
                     lower.tail=FALSE,
                     log=FALSE)
        
        numbersCt[type1, "P"] <- p1
        p2 <- phyper(n= sum(numbersCt[,"TRUE"]), m= sum(numbersCt[,"FALSE"]),
                     k=sum(pop[,"Label"]==type1), q=numbersCt[type1,"FALSE"],
                     lower.tail=FALSE,
                     log=FALSE)
        numbersCt[type1, "P(not)"] <- p2
    }

    seNum <- sum(pop[,"Population"])
    print("The ratio of SEN or non-SEN cells in each cell types:")
    print(numbersCt)

    return(list(df=df,pop=pop,output=numbersCt))

}

In [ ]:
calculate_weighted_expression_df <- function(seurat_obj, weight_df, assay = "RNA", slot = "data") {
  expression_matrix <- GetAssayData(seurat_obj, assay = assay, slot = slot)
  print(dim(expression_matrix))
  common_genes <- intersect(rownames(expression_matrix), weight_df$Name)
  
  if (length(common_genes) == 0) {
    stop("No common genes!")
  }
  
  expression_matrix <- expression_matrix[common_genes, , drop = FALSE]
  gene_weights <- setNames(weight_df$Weight, weight_df$Name)
  gene_weights <- gene_weights[common_genes]
  weighted_scores <- colSums(expression_matrix * gene_weights, na.rm = TRUE)
  
  weighted_df <- data.frame(Cell = colnames(expression_matrix), Weighted_Expression = weighted_scores, row.names = NULL)
  
  return(weighted_df)
}

### Process RNA part

In [ ]:
Labels <- setNames(obj.gage@meta.data$sub_cell_type, rownames(obj.gage@meta.data))
stages <- obj.gage$condition
gene.set.1 <- gene.set.1[gene.set.1 %in% rownames(obj.gage)]
gene.set.2 <- gene.set.2[gene.set.2 %in% rownames(obj.gage)]
gene.set.3 <- gene.set.3[gene.set.3 %in% rownames(obj.gage)]

In [ ]:
res_ad <- process_eigengene(pigengene0_ad, 0, path2res, gene.set.1, Labels, stages, 1)
res_ct <- process_eigengene(pigengene0_ct, 0, path2res, gene.set.1, Labels, stages, 1)

In [ ]:
res0 <- process_eigengene(pigengene0_rna, 0, path2res, gene.set.1, Labels, stages, 2)
res1 <- process_eigengene(pigengene1_rna, 1, path2res, gene.set.2, Labels, stages, 2)
res2 <- process_eigengene(pigengene2_rna, 2, path2res, gene.set.3, Labels, stages, 2)

In [ ]:
weight_df0 <- res0$df
weight_df1 <- res1$df
weight_df2 <- res2$df

In [ ]:
weight_ad <- res_ad$df
weight_ct <- res_ct$df


In [ ]:
merged_df <- merge(weight_ad, weight_ct, by = "Name")
colnames(merged_df) <- c("name","ad","ct")

In [ ]:
merged_df$diff <- merged_df$ad - merged_df$ct

In [ ]:
normalize_0_1 <- function(x) {
  return ((x - min(x)) / (max(x) - min(x)))
}

In [ ]:
plot_normalized_expression <- function(pop, sub_base_colors, title, thres, save_path = NULL, width = 10, height = 6, dpi = 400) {
  pop$normExpression <- normalize_0_1(pop$Expression)
  
  # cell_order <- names(sub_base_colors)
  # pop$Cell_type <- factor(pop$Cell_type, levels = cell_order)
  valid_cell_types <- names(sub_base_colors)[names(sub_base_colors) %in% unique(pop$Cell_type)]
  pop <- pop[pop$Cell_type %in% valid_cell_types, ]
  pop$Cell_type <- factor(pop$Cell_type, levels = valid_cell_types)
  sub_base_colors <- sub_base_colors[valid_cell_types]
  
  gg1 <- ggplot(pop, aes(x = Cell_type, y = normExpression, fill = Cell_type)) + 
    geom_boxplot(outlier.shape = NA, width = 0.6, alpha = 0.8) + 
    theme_minimal() +  
    theme(
      axis.text.x = element_text(size = 15, angle = 45, hjust = 1, color = "black"),  
      axis.text.y = element_text(size = 15, color = "black"),
      axis.title = element_text(size = 22, face = "bold"),
      axis.title.x = element_blank(),
      legend.position = "none",
      axis.title.y = element_text(size = 15, margin = margin(t = 0, r = 20, b = 0, l = 0))
    ) +
    labs(title = title,
         y = "Min-Max normalized senescence score") +
    scale_fill_manual(values = sub_base_colors) 
  
  e1 <- pop$normExpression
  # gg1 <- gg1 + geom_hline(yintercept = mean(e1) + thres * sd(e1), 
  #                          linetype = "dashed", col = "red", size = 1)
  
  if (!is.null(save_path)) {
    ggsave(filename = save_path, plot = gg1, dpi = dpi, width = width, height = height, units = "in")
    message("Plot saved to: ", save_path)
  }
  
  print(gg1)
}


In [ ]:
plot_normalized_expression(res0$pop, sub_base_colors, 
                           "Weighted CSP Markers Expression (GAGE-seq RNA)",
                           1, file.path(path2res, "weights_eigen_0_rna-all.pdf") )

In [ ]:
plot_normalized_expression(res2$pop, sub_base_colors, 
                           "Weighted SASP Markers Expression (GAGE-seq RNA)",
                           1, file.path(path2res, "weights_eigen_2_rna-all.pdf") )

### Process spatial part

In [ ]:
Labels <- setNames(obj.spatial@meta.data$sub_cell_type, rownames(obj.spatial@meta.data))
stages <- obj.spatial$condition
gene.set.1 <- gene.set.1[gene.set.1 %in% rownames(obj.spatial)]
gene.set.2 <- gene.set.2[gene.set.2 %in% rownames(obj.spatial)]
gene.set.3 <- gene.set.3[gene.set.3 %in% rownames(obj.spatial)]

In [ ]:
res01 <- process_eigengene(pigengene0, 0, path2res, gene.set.1, Labels, stages, 2)
res11 <- process_eigengene(pigengene1, 1, path2res, gene.set.2, Labels, stages, 2)
res21 <- process_eigengene(pigengene2, 2, path2res, gene.set.3, Labels, stages, 2)

In [ ]:
write.csv(res0$output, file = "csp_rna.csv", row.names = TRUE)
write.csv(res2$output, file = "sasp_rna.csv", row.names = TRUE)

In [ ]:
csp_rna <- read.csv("csp_rna.csv")
head(csp_rna)

In [ ]:
df <- csp_rna
df <- df %>%
  mutate(celltype = sub("-AD|-CT", "", X)) %>%
  
  group_by(celltype) %>%
  summarise(
    AD_CT_FALSE = sum(FALSE.),
    AD_CT_TRUE = sum(TRUE.),
    total = AD_CT_FALSE + AD_CT_TRUE,
    ratio = AD_CT_TRUE / total
  )
csp_rna <- df

In [ ]:
sasp_rna <- read.csv("sasp_rna.csv")
head(sasp_rna)

In [ ]:
df <- sasp_rna
df <- df %>%
  mutate(celltype = sub("-AD|-CT", "", X)) %>%
  
  group_by(celltype) %>%
  summarise(
    AD_CT_FALSE = sum(FALSE.),
    AD_CT_TRUE = sum(TRUE.),
    total = AD_CT_FALSE + AD_CT_TRUE,
    ratio = AD_CT_TRUE / total
  )
sasp_rna <- df

In [ ]:
write.csv(res01$output, file = "csp_spatial.csv", row.names = TRUE)
write.csv(res21$output, file = "sasp_spatial.csv", row.names = TRUE)

In [ ]:
csp_spatial <- read.csv("csp_spatial.csv")
csp_spatial <- csp_spatial %>% 
  filter(!grepl("Lowexpr", X))
head(csp_spatial)

In [ ]:
df <- csp_spatial
df <- df %>%
  mutate(celltype = sub("-AD|-CT", "", X)) %>%
  
  group_by(celltype) %>%
  summarise(
    AD_CT_FALSE = sum(FALSE.),
    AD_CT_TRUE = sum(TRUE.),
    total = AD_CT_FALSE + AD_CT_TRUE,
    ratio = AD_CT_TRUE / total
  )
csp_spatial <- df

In [ ]:
sasp_spatial <- read.csv("csp_spatial.csv")
sasp_spatial <- sasp_spatial %>% 
  filter(!grepl("Lowexpr", X))
head(sasp_spatial)

In [ ]:
df <- sasp_spatial
df <- df %>%
  mutate(celltype = sub("-AD|-CT", "", X)) %>%
  
  group_by(celltype) %>%
  summarise(
    AD_CT_FALSE = sum(FALSE.),
    AD_CT_TRUE = sum(TRUE.),
    total = AD_CT_FALSE + AD_CT_TRUE,
    ratio = AD_CT_TRUE / total
  )
sasp_spatial <- df

In [ ]:
res0$pop <- res0$pop[colnames(obj.gage), , drop = FALSE]
obj.gage@meta.data$expr1 <- normalize_0_1(res0$pop$Expression)
res1$pop <- res1$pop[colnames(obj.gage), , drop = FALSE]
obj.gage@meta.data$expr2 <- normalize_0_1(res1$pop$Expression)
res2$pop <- res2$pop[colnames(obj.gage), , drop = FALSE]
obj.gage@meta.data$expr3 <- normalize_0_1(res2$pop$Expression)
head(obj.gage@meta.data)

In [ ]:
csp_rna$source <- "RNA"
csp_spatial$source <- "Spatial"

combined <- bind_rows(csp_rna, csp_spatial)

combined$celltype <- factor(combined$celltype, levels = unique(combined$celltype))

rna_prop <- csp_rna %>%
  select(celltype, AD_CT_TRUE) %>%
  mutate(source = "RNA") %>%
  mutate(prop = AD_CT_TRUE / sum(AD_CT_TRUE))

spatial_prop <- csp_spatial %>%
  select(celltype, AD_CT_TRUE) %>%
  mutate(source = "Spatial") %>%
  mutate(prop = AD_CT_TRUE / sum(AD_CT_TRUE))

plot_df <- bind_rows(rna_prop, spatial_prop)
plot_df <- plot_df %>% filter(!is.na(celltype))
plot_df$celltype <- factor(plot_df$celltype, levels = names(sub_base_colors))

ggplot(plot_df, aes(x = source, y = prop, fill = celltype)) +
  geom_bar(stat = "identity", position = "stack") +
  ylab("Proportion of Senescent cells (>mean + 2std)") +
  xlab("") +
  ggtitle("Senescent Cells Proportion (CSP markers)") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  scale_fill_manual(values = sub_base_colors) + 
  theme_minimal() +
  theme(axis.text.x = element_text(size = 12),
        legend.position = "right")

ggsave("csp_cell_number_compare.pdf", width = 6, height = 4, dpi = 300)


In [ ]:
rna_long <- csp_rna %>%
  select(celltype, ratio) %>%
  filter(!is.na(celltype)) %>%
  mutate(non_ratio = 1 - ratio) %>%
  pivot_longer(cols = c(ratio, non_ratio), names_to = "type", values_to = "value")

In [ ]:
rna_ratio_only <- csp_rna %>%
  filter(!is.na(celltype)) %>%
  select(celltype, ratio) %>%
  mutate(celltype = reorder(celltype, ratio))  

ggplot(rna_ratio_only, aes(x = celltype, y = ratio, fill = celltype)) +
  geom_col(width = 0.7) +
  geom_text(aes(label = percent(ratio, accuracy = 0.1)), 
            vjust = -0.3, size = 2.5, color = "black") + 
  scale_fill_manual(values = sub_base_colors) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, max(rna_ratio_only$ratio) * 1.15)) +
  labs(
    title = "Proportion of Senescent Cells in GAGE-seq RNA per Cell Type",
    x = "Cell Type",
    y = "Senescent Cell Proportion"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    axis.title.x = element_text(margin = margin(t = 10)),
    axis.title.y = element_text(margin = margin(r = 10)),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
ggsave("csp_cell_percent_rna.pdf", width = 6, height = 4, dpi = 300)

In [ ]:
rna_ratio_only <- sasp_spatial %>%
  select(celltype, ratio) %>%
  mutate(celltype = reorder(celltype, ratio))  

ggplot(rna_ratio_only, aes(x = celltype, y = ratio, fill = celltype)) +
  geom_col(width = 0.7) +
  geom_text(aes(label = percent(ratio, accuracy = 0.1)), 
            vjust = -0.3, size = 2.5, color = "black") + 
  scale_fill_manual(values = sub_base_colors) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, max(rna_ratio_only$ratio) * 1.15)) +
  labs(
    title = "Proportion of Senescent Cells in GAGE-seq Spatial per Cell Type",
    x = "Cell Type",
    y = "Senescent Cell Proportion"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    axis.title.x = element_text(margin = margin(t = 10)),
    axis.title.y = element_text(margin = margin(r = 10)),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
ggsave("sasp_cell_percent_spatial.pdf", width = 6, height = 4, dpi = 300)

In [ ]:
sasp_rna$source <- "RNA"
sasp_spatial$source <- "Spatial"

combined <- bind_rows(sasp_rna, sasp_spatial)

combined$celltype <- factor(combined$celltype, levels = unique(combined$celltype))

rna_prop <- sasp_rna %>%
  select(celltype, AD_CT_TRUE) %>%
  mutate(source = "RNA") %>%
  mutate(prop = AD_CT_TRUE / sum(AD_CT_TRUE))

spatial_prop <- sasp_spatial %>%
  select(celltype, AD_CT_TRUE) %>%
  mutate(source = "Spatial") %>%
  mutate(prop = AD_CT_TRUE / sum(AD_CT_TRUE))

plot_df <- bind_rows(rna_prop, spatial_prop)
plot_df <- plot_df %>% filter(!is.na(celltype))
plot_df$celltype <- factor(plot_df$celltype, levels = names(sub_base_colors))

ggplot(plot_df, aes(x = source, y = prop, fill = celltype)) +
  geom_bar(stat = "identity", position = "stack") +
  ylab("Proportion of Senescent cells (>mean + 2std)") +
  xlab("") +
  ggtitle("Senescent Cells Proportion (SASP markers)") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  scale_fill_manual(values = sub_base_colors) + 
  theme_minimal() +
  theme(axis.text.x = element_text(size = 12),
        legend.position = "right")

ggsave("sasp_cell_number_compare.pdf", width = 6, height = 4, dpi = 300)

### AD senescence cells are dominated by ETS2 + CDKN1A + CDKN2D

In [ ]:
group_cells_list <- list(
  "Group 1 (AD senescence cells)" = gene_set_AD,
  "Group 2 (CT senescence cells)" = gene_set_CT,
  "Group 3 (AD non-senescence cells)" = gene_set_AD_nonsen,
  "Group 4 (CT non-senescence cells)" = gene_set_CT_nonsen
)

In [ ]:
calc_expr_percentage <- function(seurat_obj, genes, group_cells_list) {
  expr_matrix <- GetAssayData(seurat_obj, slot = "counts")
  genes_present <- genes[genes %in% rownames(expr_matrix)]

  if (length(genes_present) == 0) {
    stop("None of the input genes are found in the expression matrix.")
  }

  expr_sub <- expr_matrix[genes_present, , drop = FALSE]
  expressed_cells <- colSums(expr_sub > 0) > 0

  seurat_obj$Marker_Expression_Status <- "Not_Expressed"
  seurat_obj$Marker_Expression_Status[expressed_cells] <- "Expressed"

  expression_percentages <- sapply(group_cells_list, function(group_cells) {
    sum(seurat_obj$Marker_Expression_Status[group_cells] == "Expressed") / length(group_cells)
  })

  return(expression_percentages)
}

In [ ]:
genes.AD.sen <- c("ETS2", "CDKN1A", "CDKN2D")
# genes.AD.sen <- gene.set.1
res <- calc_expr_percentage(obj.gage, genes.AD.sen, group_cells_list)
print(res)

In [ ]:
cell_type_order <- c(
  "Exc", "Inh", "Astro", "Micro", "Oligo", "OPC", "Endo", "VLMC"
)
obj.gage$major_cell_type <- factor(
  obj.gage$major_cell_type,
  levels = rev(cell_type_order)  
)

In [ ]:
genes <- genes.AD.sen  

Idents(obj.gage) <- "major_cell_type"

p_dot <- DotPlot(
  obj.gage,
  features = genes
) + 
  RotatedAxis() +
  scale_color_gradient(
    name = "Average\nexpression",
    low  = "#F2F0F7",
    high = "#54278F"
  ) +
  scale_size(
    name = "Percent\nexpressing",
    range = c(1, 6)
  ) +
  theme_minimal(base_size = 12) +
  theme(
    panel.grid.major = element_line(linewidth = 0.2, colour = "grey90"),
    panel.grid.minor = element_blank(),
    axis.title.x     = element_blank(),
    axis.title.y     = element_blank(),
    axis.text.x      = element_text(
      angle = 45,
      hjust = 1,
      vjust = 1,
      size  = 10
    ),
    axis.text.y      = element_text(size = 11),
    legend.position  = "right",
    legend.box       = "vertical",
    legend.title     = element_text(size = 10),
    legend.text      = element_text(size = 9),
    plot.title       = element_text(hjust = 0.5, face = "bold")
  ) +
  ggtitle("Expression of selected genes across cell types")

p_dot

## DEG calling

In [ ]:
obj.gage$sen_group <- "Other"
cell_names <- rownames(seurat_obj@meta.data)

obj.gage$sen_group[cell_names %in% gene_set_AD] <- "AD-sen"
obj.gage$sen_group[cell_names %in% gene_set_CT] <- "CT-sen"
obj.gage$sen_group[cell_names %in% gene_set_AD_nonsen] <- "AD-nonsen"
obj.gage$sen_group[cell_names %in% gene_set_CT_nonsen] <- "CT-nonsen"

In [ ]:
path2degresdir <- "/home/xinyuelu/AD-proj/scripts_results/annotate_cells/senescence"

In [ ]:
f = function(ident.1, ident.2, min.pct, only.pos, filename) {
    print(filename)
    ident.2 = ident.2[!(ident.2 %in% ident.1)]
    df = FindMarkers(
        obj, min.pct=min.pct, only.pos=only.pos, test.use='MAST', verbose=F,
        ident.1=ident.1, ident.2=ident.2,
    )
    write.csv(df, file.path(path2degresdir, paste(filename, '.csv', sep='')))
    return(df)
}

In [ ]:
obj <- subset(obj.gage, subset = major_cell_type == 'Micro')
Idents(obj) <- obj$sen_group

In [ ]:
table(obj@meta.data$sen_group, obj@meta.data$sex)